In [4]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

In [5]:
#Load Config
CONFIG_PATH = os.path.join("config","config.json")
TRANSCRIPT_PATH = os.path.join("data","transcripts.xlsx")
config = None
with open(CONFIG_PATH, 'r') as reader:
    config = json.load(reader)

#Load Transcripts
transcript_df = pd.read_excel(TRANSCRIPT_PATH)

In [6]:
#Load LLM
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def get_llm(config: str):
    provider = os.getenv("LLM_PROVIDER")
    
    if provider == 'openai':
        return ChatOpenAI(
            model=config['llm']['openai_model'],
            temperature=config['llm']['temperature']
        )
    elif provider == 'gemini':
        return ChatGoogleGenerativeAI(
            model=config['llm']['gemini_model'],
            temperature=config['llm']['temperature']
        )
    else:
        raise ValueError(f"Invalid LLM Provider. Please provide appropriate LLM_PROVIDER: {provider}")

llm = get_llm(config)
print(f"LLM initialized using the provider: {os.getenv('LLM_PROVIDER')}")

LLM initialized using the provider: openai


In [ ]:
#Defining the Output Schema
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ClassificationOutput(BaseModel):
    call_type: str = Field(description="Type of customer call")
    confidence: float = Field(description="Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

prompt = PromptTemplate(
    template="""
    You are a call classification assistant.
    Classify the following customer support transcript into one of these categories:
    {labels}
    
    Transcript:
    {transcript}
    
    {format_instructions}
    """,
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)

classification_chain = prompt | llm | parser
sample_Text = transcript_df.iloc[0]["transcript"]
response = classification_chain.invoke({"transcript": sample_Text})
print(f"Classification Result:\n{response}")

Classification Result:
billing


In [10]:
from tqdm import tqdm
results = []

for i,row in tqdm(transcript_df.iterrows(), total=len(transcript_df), desc="Classification of the Calls"):
    try:
        classification = classification_chain.invoke({
        "transcript": row["transcript"]
        })
        
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": classification.call_type,
            "confidence": classification.confidence
        })
    except Exception as e:
        print(f"Error at the row {i}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

results_df = pd.DataFrame(results)
transcript_df = transcript_df.merge(results_df, on="call_id")
transcript_df.head(2)

Classification of the Calls: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]


,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Matt,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Kim,Customer: My claim has been pending for 2 week...,claims,claims,0.95


In [12]:
#Define Routing Logic
def route_call(call_type):
    if call_type=="billing":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type=="claims":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type=="complaint":
        return ["tone_empathy", "resolution_quality"]
    elif call_type=="general_query":
        return ["knowledge_accuracy"]
    else: 
        return ["knowledge_accuracy"]

transcript_df["evaluation_plan"] = transcript_df["predicted_call_type"].apply(route_call)
transcript_df.head(2)

,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence,evaluation_plan
0,1,Matt,Customer: I was charged twice for my premium t...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]"
1,2,Kim,Customer: My claim has been pending for 2 week...,claims,claims,0.95,"[knowledge_accuracy, resolution_quality]"


In [13]:
class ToneEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object=ToneEvaluation)

tone_prompt = PromptTemplate(
    template="""
    You are a QA evaluator for customer support calls.
    
    Evaluate the agent's tone and empathy in the following transcript.
    
    Consider:
    - Did the agent acknowledge the customer's issue?
    - Was the tone polite and professional?
    - Did the agent show empathy?
    
    Transcript:
    {transcript}
    
    {format_instructions}
    """,
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": tone_parser.get_format_instructions()
    }
)

tone_chain = tone_prompt | llm | tone_parser

response = tone_chain.invoke({"transcript": sample_Text})
print(f"Tone Evaluation\n{response}")

Tone Evaluation
score=3 reasoning="The agent acknowledged the customer's issue by stating they would check the charge, which indicates some level of engagement. However, the tone is neutral and lacks warmth or empathy, as there are no expressions of understanding or concern for the customer's situation. The response is polite but could benefit from a more empathetic approach."


In [14]:
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

resolution_prompt = PromptTemplate(
    template="""
    You are a QA evaluator for customer support calls.
    
    Evaluate the resolution quality of the agent.
    
    Consider:
    - Did the agent fully resolve the customer's issue?
    - Were next steps clearly communicated?
    - Did the agent confirm resolution before ending?
    
    Transcript:
    {transcript}
    
    {format_instructions}
    """,
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

resolution_chain = resolution_prompt | llm | resolution_parser

response = resolution_chain.invoke({"transcript": sample_Text})
print(f"Resolution Evaluation\n{response}")

Resolution Evaluation
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge or communicated any follow-up actions."


In [15]:
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

knowledge_prompt = PromptTemplate(
    template="""
    You are a QA evaluator for customer support calls.
    
    Evaluate the agent's knowledge accuracy and clarity.
    
    Consider:
    - Did the agent provide correct and relevant information?
    - Was the explanation clear and easy to understand?
    - Did the agent avoid vague or misleading statements?
    
    IMPORTANT:
    - If the transcript does not contain enough information, give a moderate score (2 or 3) and explain what is missing.
    Transcript:
    {transcript}
    
    {format_instructions}
    """,
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

knowledge_chain = knowledge_prompt | llm | knowledge_parser

response = knowledge_chain.invoke({"transcript": sample_Text})
print(f"Knowledge Evaluation\n{response}")

Knowledge Evaluation
score=2 reasoning="The transcript lacks sufficient information to evaluate the agent's knowledge accuracy and clarity. The agent only initiated a check without providing any specific information or resolution regarding the double charge issue. Therefore, it's unclear if the agent is knowledgeable about the situation or if the explanation would have been clear and relevant."


In [16]:
#Evaluation Runner
def run_evaluation(transcript, eval_plan):
    results = {}
    
    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"Error": str(e)}
            
    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"Error": str(e)}
            
    if "resolution_quality" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"Error": str(e)}
    
    return results

#Apply to the entire dataset
evaluation_outputs = []

for i,row in tqdm(transcript_df.iterrows(), total=len(transcript_df), desc="Running the Evaluation"):
    try:
        evaluation = run_evaluation(row["transcript"], row["evaluation_plan"])
        evaluation_outputs.append({
            "call_id": row["call_id"],
            "evaluation_output": evaluation
        })
    except Exception as e:
        evaluation_outputs.append({
            "call_id": row["call_id"],
            "evaluation_output": None
        })

eval_df = pd.DataFrame(evaluation_outputs)
transcript_df = transcript_df.merge(eval_df, on="call_id")
transcript_df.head(2)

Running the Evaluation: 100%|██████████| 4/4 [00:13<00:00,  3.44s/it]


,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence,evaluation_plan,evaluation_output
0,1,Matt,Customer: I was charged twice for my premium t...,billing,billing,0.95,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 2, 'reasoning': 'The t..."
1,2,Kim,Customer: My claim has been pending for 2 week...,claims,claims,0.95,"[knowledge_accuracy, resolution_quality]","{'knowledge': {'score': 2, 'reasoning': 'The t..."


In [17]:
#Final Report Schema
class FinalReportSchema(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendation: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReportSchema)

final_prompt = PromptTemplate(
    template="""
    You are a QA manager reviewing customer support calls.
    
    Based on the evaluation results below, generate:
    1. A concise summary of the agent's performance
    2. A list of actionable recommendations for improvement
    
    Evaluation Data:
    {evaluation_output}
    
    IMPORTANT:
    - Be specfic and practical
    - Do not repeat scores
    - Focus on improvement
    
    {format_instructions}
    """,
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

final_chain = final_prompt | llm | final_parser
sample_eval = transcript_df.iloc[0]["evaluation_output"]
response = final_chain.invoke({"evaluation_output": sample_eval})
print(f"Final QA Report:\n{response}")

Final QA Report:
summary="The agent demonstrated insufficient knowledge and failed to resolve the customer's issue regarding a double charge. The conversation lacked clarity and completeness, leaving the customer without a clear understanding of the next steps." recommendation=['Enhance training on common billing issues, specifically double charges, to improve knowledge and response accuracy.', "Encourage agents to provide detailed explanations and confirm understanding of the customer's issue before proceeding with next steps.", 'Implement a protocol for agents to summarize the resolution process and confirm with the customer that their issue has been addressed before ending the call.', 'Conduct regular role-playing exercises to practice handling complex customer inquiries and ensuring complete resolutions.']


In [20]:
final_outputs = []

for i,row in tqdm(transcript_df.iterrows(), total=len(transcript_df), desc="Final Report Summary Generation"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendation
        })
    except Exception as e:
        print(f"Error at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })

final_report_df = pd.DataFrame(final_outputs)
transcript_df = transcript_df.merge(final_report_df, on="call_id")

Final Report Summary Generation: 100%|██████████| 4/4 [00:13<00:00,  3.26s/it]


In [21]:
transcript_df.to_excel("data/output.xlsx", index=False)